In [1]:
# project 1st file 기본 전처리(robustScaler) 후 모델 돌려보기 2025.11.27 SVM 만 
import sys

project_root = 'c:/big20/git/big20-ML-project2-team3/CreditCardFraud'

# sys.path에 추가 (모듈 import용)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import os

import importlib
from utils import user_utils
importlib.reload(user_utils)

import utils.preprocessing as pp 
import utils.data_sampling as ds 
import utils.user_utils    as uu 

c:\ProgramData\anaconda3\envs\ml_dev\Lib\site-packages\hyperopt\atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# data loading
df = pp.ccf_load_data()

데이터 로드 성공: (284807, 31)


In [ ]:
# 기본정보 확인
pp.checkBasicInfo(df, isInfo=False, isGraph=True)


In [ ]:
# 상관관계 시각화 시도
pp.corrGraph(df)    

In [ ]:
# X_features, y_target = pp.split_features_target(df, cols='Time', target_col='Class')
# df_desc_org = uu.get_outlier(X_features)


In [ ]:
# df_desc_org
# outlier_cols = df_desc_org[df_desc_org['OutlierCount'] > 10000].index.to_list()[:-1]
# outlier_cols

In [ ]:
# df['Amount'].value_counts()
# df['Amount'].value_counts(normalize=True)
# df[df['Amount']> 184]['Amount']

In [4]:
# 2. Data전처리
# Amount와 Time 스케일링 (V1-V28은 이미 PCA 처리됨)
df_robust_scaled = pp.robustScaler(df)

✅ 'Amount' 스케일링 완료 → 'Amount_scaled'
✅ 'Time' 스케일링 완료 → 'Time_scaled'

🗑️  원본 컬럼 제거: ['Amount', 'Time']

✅ 스케일링 완료: 2개 컬럼
   최종 shape: (284807, 31)


In [5]:
## 데이터 
X_features, y_target = pp.split_features_target(df_robust_scaled)
X_train, X_test, y_train, y_test = pp.data_split(X_features, y_target)

In [6]:
# Over Sampling
X_over, y_over = ds.oversampling_smote(X_train, y_train)

✅ SMOTE 오버샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)


In [7]:
# Under Sampling
X_under, y_under = ds.undersampling_RUS(X_train, y_train)

✅ 랜덤 언더샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 788 (Class 0: 394, Class 1: 394)
   제거된 샘플: 227057개


In [8]:
# Combined Sampling
X_combined, y_combined = ds.combined_sampling(X_train, y_train)

✅ SMOTETomek 혼합 샘플링 완료
   원본 샘플 수: 227845 (Class 0: 227451, Class 1: 394)
   샘플링 후: 454902 (Class 0: 227451, Class 1: 227451)
   최종 변화: +227057개


In [85]:
# outlier_index = get_outlier_index(df_, column=outlier_cols)
# print(f"outlier_index cnt: {len(outlier_index)}, rate:{(len(outlier_index) / df_sacled.shape[0]) * 100}%")
# outlier_index cnt: 492, rate:0.1727485630620034%

In [11]:
option_name = 'robustScaler_HO'

In [8]:
results = {}


In [ ]:
from sklearn.svm     import SVC
from sklearn.metrics import classification_report

# 5. Linear SVM
svc_liner_basic_params = {
  'C' : 1.0, 
  'kernel' : "linear", 
  'class_weight' : "balanced"
}
linear_svm = SVC(**svc_liner_basic_params)  # 불균형 데이터라 balanced 권장
resutls_scr='''
✓ 모델 저장 완료: ../models\lsvm_robustScaler_HO.pkl
  파일 크기: 6.25 MB
'''

# results[f'lsvm_{option_name}'] = uu.get_model_train_eval(linear_svm, f'lsvm_{option_name}', X_train,X_test,y_train,y_test)

linear_svm.fit(X_train, y_train)
y_pred_linear = linear_svm.predict(X_test)

print("=== Linear SVM ===")
print(classification_report(y_test, y_pred_linear, digits=4))



=== Linear SVM ===
              precision    recall  f1-score   support

           0     0.9998    0.9777    0.9886     56864
           1     0.0642    0.8878    0.1198        98

    accuracy                         0.9775     56962
   macro avg     0.5320    0.9327    0.5542     56962
weighted avg     0.9982    0.9775    0.9871     56962



: 

In [ ]:
from utils.model_utils import save_model
model_name_lsvm = 'lsvm_robustScaler_basic'
save_model(linear_svm, model_name_lsvm)
results = {}
# report = classification_report(y_test, y_pred_linear, digits=4, output_dict=True)
# results_df = pd.DataFrame(report).T
result_txt = '''
              precision    recall  f1-score   support
           0     0.9998    0.9777    0.9886     56864
           1     0.0642    0.8878    0.1198        98
    accuracy                         0.9775     56962
   macro avg     0.5320    0.9327    0.5542     56962
weighted avg     0.9982    0.9775    0.9871     56962
'''

위 결과 해석 by colpilot
---

## 📊 결과 해석 (Linear SVM)

### 클래스별 성능
- **클래스 0 (정상 데이터, 56,864개)**
  - **Precision 0.9998** → 모델이 "0"이라고 예측한 것 중 거의 모두가 실제로 0임 (거의 완벽).
  - **Recall 0.9777** → 실제 0 중 약 97.8%를 잘 맞춤. 일부(약 2.2%)는 놓쳤음.
  - **F1-score 0.9886** → Precision과 Recall이 모두 높아 균형 잡힌 성능.

- **클래스 1 (이상 데이터, 98개)**
  - **Precision 0.0642** → 모델이 "1"이라고 예측한 것 중 실제 1은 6.4%밖에 안 됨. 즉, **거짓 양성(False Positive)**이 매우 많음.
  - **Recall 0.8878** → 실제 1 중 88.8%를 잘 잡아냄. 놓친 건 11.2% 정도.
  - **F1-score 0.1198** → Precision이 너무 낮아서 F1도 낮음. **많이 잡긴 하지만 정확성이 떨어짐**.

---

### 전체 성능
- **Accuracy 0.9775 (97.75%)**  
  전체적으로는 매우 높은 정확도. 하지만 데이터가 **불균형(0이 압도적으로 많음)**이라 accuracy만 보면 착각할 수 있음.

- **Macro avg (0.5320 / 0.9327 / 0.5542)**  
  클래스별 성능을 단순 평균한 값. Precision은 낮지만 Recall은 높음 → **불균형 데이터에서 성능 차이가 큼**을 보여줌.

- **Weighted avg (0.9982 / 0.9775 / 0.9871)**  
  데이터 개수 비율을 반영한 평균. 클래스 0이 많아서 전체 평균이 매우 높게 나옴. → **불균형 데이터의 영향**.

---

## 🧩 종합 해석
- 모델은 **클래스 0(정상)**을 거의 완벽하게 맞추지만,  
- 클래스 1(이상)은 **많이 잡아내긴 하지만 Precision이 매우 낮아** "이상이라고 예측한 것 중 실제 이상은 거의 없음".  
- 즉, **이상 탐지에서 False Positive가 심각하게 많다**는 뜻입니다.  

---

👉 결론적으로, 이 모델은 **정상 데이터 분류는 잘하지만 이상 데이터 분류는 정확성이 떨어지는 불균형 문제**가 있습니다.  
이를 개선하려면:
- 클래스 불균형을 완화하는 기법 (예: SMOTE, 언더샘플링, 클래스 가중치 조정)  
- 다른 알고리즘 (예: Random Forest, XGBoost)  
- Decision threshold 조정  

등을 고려할 수 있습니다.  

In [ ]:
# classification_report 결과를 dict 형태로 입력
report = {
    '0': {'precision': 0.9998, 'recall': 0.9777, 'f1-score': 0.9886, 'support': 56864},
    '1': {'precision': 0.0642, 'recall': 0.8878, 'f1-score': 0.1198, 'support': 98},
}

# 클래스와 지표
classes = ['0', '1']
metrics = ['precision', 'recall', 'f1-score']
colors = ['#8884d8', '#82ca9d', '#ff7c7c']

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots(figsize=(10,6))

for i, metric in enumerate(metrics):
    values = [report[c][metric] for c in classes]
    ax.bar(x + i*width - width, values, width, label=metric, color=colors[i], alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim([0,1])
ax.set_ylabel('Score')
ax.set_title('Linear SVM Classification Report (Class 0 vs Class 1)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# results[f'lsvm_{option_name}_smote'] = uu.get_model_train_eval(linear_svm, f'lsvm_{option_name}_smote', X_over,X_test,y_oever,y_test)

In [ ]:
# results[f'lsvm_{option_name}_under'] = uu.get_model_train_eval(linear_svm, f'lsvm_{option_name}_under', X_under,X_test,y_under,y_test)


In [ ]:
# results[f'lsvm_{option_name}_combined'] = uu.get_model_train_eval(linear_svm, f'lsvm_{option_name}_combined', X_combined,X_test,y_combined,y_test)

In [ ]:
# 6. RBF SVM
svc_rbf_basic_params = {
  'C' : 1.0, 
  'kernel' : "rbf", 
  'gamma' : "scale",
  'class_weight' : "balanced"  
}
rbf_svm = SVC(**svc_rbf_basic_params)
# rbf_svm.fit(X_train, y_train)
# y_pred_rbf = rbf_svm.predict(X_test)

# print("=== RBF SVM ===")
# print(classification_report(y_test, y_pred_rbf, digits=4))



In [ ]:
df_results = pd.DataFrame(results).T